# Notebook 02 — Build the custom reference (genome + transgene contig)

**Project:** DE_project — snRNA-seq transgene detection in *Arabidopsis* (Col-0)

produce a single, documented, reproducible reference that the snRNA-seq
reads can be aligned to — the Col-0 genome with the `pFusionMYB41` transgene contig
added as an extra sequence, plus one matching annotation file.

**Base genome:** `Athaliana.Col-0.HPIv02` (Salk HPI). *(Note: HPIv01 and HPIv02 have
byte-identical genome + gene models; v02 only adds functional annotation. We use v02.)*

**Steps**
1. Fix the malformed transgene contig FASTA (it has no `>` header).
2. Append the contig to the genome → combined genome FASTA (8 → 9 sequences).
3. Reconcile annotations: convert the base **GFF3 → GTF**, then merge with the contig GTF.
4. Validate the combined reference (contig present, Scar 2 in place, native + fusion MYB41).
5. *(Optional)* build the aligner index — STARsolo vs Cell Ranger (pending platform).

Everything downstream depends on this being correct, so each step is checked.


## Step 0 — Setup, paths, tools

In [6]:
import os, sys, gzip, shutil, subprocess
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

PROJ = "/data1/bfernando/DE_project"
REF  = os.path.join(PROJ, "reference")
BUILD = os.path.join(REF, "build")
os.makedirs(BUILD, exist_ok=True)

# --- inputs ---
GENOME_GZ  = os.path.join(REF, "Athaliana.Col-0.HPIv02.genome.fasta.gz")   # base genome
GENE_GFF3  = os.path.join(REF, "Athaliana.Col-0.HPIv02.gene.gff3.gz")       # base annotation (GFF3)
CONTIG_RAW = os.path.join(PROJ, "Athaliana.Col-0.HPIv01.pFusionMYB41_only.fasta")  # malformed
CONTIG_GTF = os.path.join(PROJ, "Athaliana.Col-0.HPIv01.pFusionMYB41_only.gtf")    # contig annotation (GTF)

# --- outputs ---
CONTIG_FA     = os.path.join(BUILD, "pFusionMYB41.fasta")                       # fixed contig
COMBINED_FA   = os.path.join(BUILD, "Athaliana.Col-0.HPIv02.plus_pFusionMYB41.fasta")
BASE_GTF      = os.path.join(BUILD, "Athaliana.Col-0.HPIv02.gene.gtf")          # GFF3 -> GTF
COMBINED_GTF  = os.path.join(BUILD, "Athaliana.Col-0.HPIv02.plus_pFusionMYB41.gtf")

# --- tools: resolve from THIS env's bin so subprocess finds them ---
BIN = os.path.dirname(sys.executable)
GFFREAD  = os.path.join(BIN, "gffread")
SAMTOOLS = os.path.join(BIN, "samtools")
STAR     = os.path.join(BIN, "STAR")

for p in (GENOME_GZ, GENE_GFF3, CONTIG_RAW, CONTIG_GTF):
    print(("OK   " if os.path.exists(p) else "MISS ") + os.path.basename(p))
print("tools:", all(os.path.exists(t) for t in (GFFREAD, SAMTOOLS, STAR)))


OK   Athaliana.Col-0.HPIv02.genome.fasta.gz
OK   Athaliana.Col-0.HPIv02.gene.gff3.gz
OK   Athaliana.Col-0.HPIv01.pFusionMYB41_only.fasta
OK   Athaliana.Col-0.HPIv01.pFusionMYB41_only.gtf
tools: True


## Step 1 — Fix the malformed contig FASTA

The provided `...pFusionMYB41_only.fasta` is **not valid FASTA** — it has no `>` header;
it's a single line of `pFusionMYB41<TAB><sequence>`. 
Storing it in correct format


In [7]:
def read_fasta_robust(path):
    """Return (name, seq). Handles proper FASTA AND the headerless name<TAB>seq quirk."""
    text = open(path).read()
    if text.lstrip().startswith(">"):
        rec = next(SeqIO.parse(path, "fasta"))
        return rec.id, str(rec.seq).upper()
    # headerless: first whitespace/tab splits name from sequence
    first = text.splitlines()[0]
    name, seq = first.split(None, 1)
    seq = "".join(seq.split()).upper()     # strip any internal whitespace
    return name, seq

name, seq = read_fasta_robust(CONTIG_RAW)
assert set(seq) <= set("ACGTN"), f"unexpected characters: {set(seq) - set('ACGTN')}"

rec = SeqRecord(Seq(seq), id=name, description="")
SeqIO.write(rec, CONTIG_FA, "fasta")

print(f"contig name        : {name}")
print(f"contig length      : {len(seq)} bp")
print(f"written proper FASTA: {CONTIG_FA}")
print("first 2 lines of the fixed file:")
print("  " + "\n  ".join(open(CONTIG_FA).read().splitlines()[:2]))


contig name        : pFusionMYB41
contig length      : 1866 bp
written proper FASTA: /data1/bfernando/DE_project/reference/build/pFusionMYB41.fasta
first 2 lines of the fixed file:
  >pFusionMYB41
  CAACTATGTATAATAAAGTTGCTGGCCAAATTAACGGCTTTAAAATGAACTTGAATTGGG


## Step 2 — Append the contig to the genome

Decompress the genome and concatenate the fixed contig onto the end → one combined
FASTA.

**The original content** (7 sequences that came with the download):

| Name | What it is | Size (from our `.fai`) |
|---|---|---|
| Chr1–Chr5 | the 5 nuclear chromosomes | 18–30 Mb each |
| ChrC | chloroplast genome | 154,478 bp |
| ChrM | mitochondrial genome | 366,924 bp |

The genome has 7 sequences (Chr1–5, ChrC, ChrM); after appending the transgene contig the result should have **8**.



In [8]:
with open(COMBINED_FA, "w") as out:
    with gzip.open(GENOME_GZ, "rt") as g:      # genome (decompressed)
        shutil.copyfileobj(g, out)             # streams all of g into out.
    with open(CONTIG_FA) as c:                 # + transgene contig
        shutil.copyfileobj(c, out)

# report every sequence + length in the combined reference
lengths = {}                                        
for r in SeqIO.parse(COMBINED_FA, "fasta"):         # for each sequence in the file
    lengths[r.id] = len(r.seq)                      # record  name -> length

print(f"Combined FASTA: {COMBINED_FA}")
print(f"  {len(lengths)} sequences:")

for cid, L in lengths.items():
    if cid == name:
        tag = "transgene contig"
    else: 
        tag = ""
    print(f"    {cid:16} {L:>11,} bp{tag}")
assert name in lengths and len(lengths) == 8, "expected 8 sequences incl. the contig!"
print("OK: contig is present as the 8th sequence.")


Combined FASTA: /data1/bfernando/DE_project/reference/build/Athaliana.Col-0.HPIv02.plus_pFusionMYB41.fasta
  8 sequences:
    Chr1              30,427,671 bp
    Chr2              19,698,289 bp
    Chr3              23,459,830 bp
    Chr4              18,585,056 bp
    Chr5              26,975,502 bp
    ChrC                 154,478 bp
    ChrM                 366,924 bp
    pFusionMYB41           1,866 bptransgene contig
OK: contig is present as the 8th sequence.


## Step 3 — Reconcile annotations (GFF3 → GTF) and merge

The base annotation is **GFF3** (phytozome style); the contig annotation is **GTF**
(AGAT style). We convert the base to GTF with `gffread -T`, then concatenate the contig
GTF so the whole reference has one consistent annotation file.


In [ ]:
import os 

# gffread can't read .gz directly -> decompress base GFF3 first
base_gff3_plain = os.path.join(BUILD, "Athaliana.Col-0.HPIv02.gene.gff3")
with gzip.open(GENE_GFF3, "rt") as g, open(base_gff3_plain, "w") as o:
    shutil.copyfileobj(g, o)

# GFF3 -> GTF
subprocess.run([GFFREAD, base_gff3_plain, "-T", "--force-exons", "-o", BASE_GTF], check=True)
# --force-exons: source GFF3 has NO exon features (only CDS/UTR); without this,
# ~3,824 CDS-only transcripts get no exon line and Cell Ranger mkref rejects the GTF.

# merge base GTF + contig GTF
with open(COMBINED_GTF, "w") as out:
    for f in (BASE_GTF, CONTIG_GTF):
        with open(f) as fh:
            shutil.copyfileobj(fh, out)
        out.write("\n")

def gtf_stats(path):
    seqnames, genes, feats = set(), set(), 0
    for ln in open(path):
        if ln.startswith("#") or not ln.strip(): # skipping comments 
            continue
        c = ln.split("\t") # split by tab and a gtf file has 9 cols else malformed.
        if len(c) < 9: 
            continue
        feats += 1
        seqnames.add(c[0]) # chr number
        for field in c[8].split(";"):
            field = field.strip()
            if field.startswith("gene_id"):
                genes.add(field.split('"')[1])
    return seqnames, genes, feats
    
print(f"base GTF     : {BASE_GTF}")
print(f"combined GTF : {COMBINED_GTF}")
print("combined GTF: has 'pFusionMYB41' seqname?",
      any(l.startswith("pFusionMYB41") for l in open(COMBINED_GTF) if not l.startswith("#")))

# (1 gene + 2 mRNAs + 5 exons + 2 UTRs + ... + 2 insertions) - 13 feature lines will be added to the GTF
for label, path in [("base GTF", BASE_GTF), ("combined GTF", COMBINED_GTF)]:
    seqs, genes, feats = gtf_stats(path)
    print(f"{label:12}: {feats:>7,} features | {len(seqs):>2} seqnames | {len(genes):,} genes")
    print("contig added? ", "pFusionMYB41" in gtf_stats(COMBINED_GTF)[0],"| fusion gene? ", "AT4G28110.Fusion" in gtf_stats(COMBINED_GTF)[1])

base GTF     : /data1/bfernando/DE_project/reference/build/Athaliana.Col-0.HPIv02.gene.gtf
combined GTF : /data1/bfernando/DE_project/reference/build/Athaliana.Col-0.HPIv02.plus_pFusionMYB41.gtf
combined GTF: has 'pFusionMYB41' seqname? True
base GTF    : 628,615 features |  7 seqnames | 27,655 genes
contig added?  True | fusion gene?  True
combined GTF: 628,628 features |  8 seqnames | 27,656 genes
contig added?  True | fusion gene?  True


## Step 4 — Validate the combined reference

Confirm the important things are all true before we ever align a read:
- the contig is present and the right length,
- **Scar 2** sits where we expect on the contig (and matches its known sequence),
- both the **native** MYB41 (`AT4G28110`) and the **fusion** (`AT4G28110.Fusion`) are annotated,
- `samtools faidx` can index the combined FASTA


In [14]:
# 4a. contig length
# checking if the gene is present in the combined file and then printing its length
contig_seq = None
for r in SeqIO.parse(COMBINED_FA, "fasta"):   # go through each sequence
    if r.id == name:                          # is this the contig ("pFusionMYB41")?
        contig_seq = str(r.seq).upper()       # grab its sequence, as an uppercase string
        break                               
print(f"[1] contig length in combined FASTA: {len(contig_seq)} bp  (expect 1866)")


# 4b. Scar 2 on the contig: plasmid 8024-8051 -> contig 390-417 (offset 7634)
scar2_expected = "AGGCCACTTTGTACAAGAAAGCTGGGTC" # from the runbook 1
scar2_on_contig = contig_seq[389:417]
print(f"[2] Scar 2 (contig 390-417): {scar2_on_contig}")
print(f"    matches known Scar 2 sequence? {scar2_on_contig == scar2_expected}")

# 4c. check if the native MYB41 and the fusion MYB41 both are in the annotation file
gtf_txt = open(COMBINED_GTF).read()
# native MYB41 lines: mention AT4G28110, but NOT the fusion, and no comment line
native = 0
for l in gtf_txt.splitlines():
    if "AT4G28110" in l and "Fusion" not in l and not l.startswith("#"):
        native += 1
        print("  " + l[:120])

# fusion MYB41 lines: mention AT4G28110.Fusion
fusion = 0
for l in gtf_txt.splitlines():
    if "AT4G28110.Fusion" in l:
        fusion += 1
        print("  " + l[:120])

print(f"\nnative lines: {native} | fusion lines: {fusion}")
print(f"[3] native MYB41 (AT4G28110) annotation lines : {native}")
print(f"    fusion MYB41 (AT4G28110.Fusion) lines      : {fusion}")

# 4d. samtools faidx must succeed
subprocess.run([SAMTOOLS, "faidx", COMBINED_FA], check=True) # run samtools as sub process
fai = COMBINED_FA + ".fai"
print(f"[4] samtools faidx OK -> {os.path.basename(fai)}")
print("    .fai contents:")
for ln in open(fai):
    print("      " + ln.strip())

assert scar2_on_contig == scar2_expected, "Scar 2 sequence mismatch on contig!"
assert native > 0 and fusion > 0, "native and/or fusion MYB41 missing from GTF!"
print("\nALL CHECKS PASSED — the combined reference is ready.")


[1] contig length in combined FASTA: 1866 bp  (expect 1866)
[2] Scar 2 (contig 390-417): AGGCCACTTTGTACAAGAAAGCTGGGTC
    matches known Scar 2 sequence? True
  Chr4	phytozomev12	transcript	13968029	13969516	.	-	.	transcript_id "AT4G28110.2.Araport11.447"; gene_id "AT4G28110.Arapo
  Chr4	phytozomev12	exon	13968029	13969143	.	-	.	transcript_id "AT4G28110.2.Araport11.447"; gene_id "AT4G28110.Araport11.4
  Chr4	phytozomev12	exon	13969252	13969516	.	-	.	transcript_id "AT4G28110.2.Araport11.447"; gene_id "AT4G28110.Araport11.4
  Chr4	phytozomev12	CDS	13968166	13968762	.	-	0	transcript_id "AT4G28110.2.Araport11.447"; gene_id "AT4G28110.Araport11.44
  Chr4	phytozomev12	transcript	13968029	13969525	.	-	.	transcript_id "AT4G28110.1.Araport11.447"; gene_id "AT4G28110.Arapo
  Chr4	phytozomev12	exon	13968029	13968751	.	-	.	transcript_id "AT4G28110.1.Araport11.447"; gene_id "AT4G28110.Araport11.4
  Chr4	phytozomev12	exon	13969014	13969143	.	-	.	transcript_id "AT4G28110.1.Araport11.447"; gene_id "AT4

## Step 5 — (Optional) Build the aligner index

**This depends on how the snRNA-seq was generated** — decide before running:
- **10x Genomics** (droplet) → build a **Cell Ranger** reference with `cellranger mkref`.
- **plate-based / other** → build a **STAR / STARsolo** index (shown below).

The STAR cell is **gated off** by default (`BUILD_INDEX = False`) so this notebook stays
fast and platform-agnostic. Flip it to `True` once the platform is confirmed. Note
`--genomeSAindexNbases 12` is the recommended value for the small (~120 Mb) *Arabidopsis*
genome (STAR warns if it's left at the default 14).


In [6]:
BUILD_INDEX = False   # set True once the sequencing platform is confirmed as STAR/STARsolo

STAR_DIR = os.path.join(BUILD, "star_index_HPIv02_pFusionMYB41")

if BUILD_INDEX:
    os.makedirs(STAR_DIR, exist_ok=True)
    cmd = [STAR, "--runMode", "genomeGenerate",
           "--genomeDir", STAR_DIR,
           "--genomeFastaFiles", COMBINED_FA,
           "--sjdbGTFfile", COMBINED_GTF,
           "--sjdbOverhang", "89",          # readlength-1; adjust to your reads
           "--genomeSAindexNbases", "12",   # for ~120 Mb genome
           "--runThreadN", "8"]
    print("running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("STAR index built at", STAR_DIR)
else:
    print("STAR index build SKIPPED (BUILD_INDEX = False).")
    print("For 10x data, build a Cell Ranger reference instead, e.g.:")
    print(f"  cellranger mkref --genome=HPIv02_pFusionMYB41 \\")
    print(f"      --fasta={COMBINED_FA} \\")
    print(f"      --genes={COMBINED_GTF}")


STAR index build SKIPPED (BUILD_INDEX = False).
For 10x data, build a Cell Ranger reference instead, e.g.:
  cellranger mkref --genome=HPIv02_pFusionMYB41 \
      --fasta=/data1/bfernando/DE_project/reference/build/Athaliana.Col-0.HPIv02.plus_pFusionMYB41.fasta \
      --genes=/data1/bfernando/DE_project/reference/build/Athaliana.Col-0.HPIv02.plus_pFusionMYB41.gtf


## Outputs

Written to `reference/build/`:

| File | What it is |
|---|---|
| `pFusionMYB41.fasta` | the transgene contig, now valid FASTA |
| `Athaliana.Col-0.HPIv02.plus_pFusionMYB41.fasta` | **combined genome** (8 sequences) + `.fai` |
| `Athaliana.Col-0.HPIv02.gene.gtf` | base annotation converted GFF3 → GTF |
| `Athaliana.Col-0.HPIv02.plus_pFusionMYB41.gtf` | **combined annotation** (native + transgene) |

**Next (notebook 03):** once the snRNA-seq FASTQs + sample sheet arrive and the platform
is confirmed, build the index (Step 5) and align — then look for reads over Scar 2
(contig 390-417) to detect and localize transgene expression.
